In [1]:
!pip install torch torchvision

In [2]:
import torch
import torchvision
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
import torch.nn as nn  # All neural network modules
from torch.utils.data import DataLoader, random_split
from torchsummary import summary
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt

Setting the device
-

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

Pretrained Model
-

resnet18 & resnet50 has already learned to identify edges, textures, and shapes from millions of images in ImageNet. We have 7*7 512 images

In [4]:
num_classes = 4 # 4 categories
model = torchvision.models.resnet50(weights="IMAGENET1K_V1") # weights = DEFAULT
# model.avgpool = nn.Identity()
model.fc = nn.Sequential(
    nn.Linear(2048, 100),  # 2048(7*7*512) if using resnet50 & 512 for resnet18
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)
model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\dbda.STUDENTSDC/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 60.0MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [5]:
model.maxpool

MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)

In [6]:
summary(model,(3,24,24))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 12, 12]           9,408
       BatchNorm2d-2           [-1, 64, 12, 12]             128
              ReLU-3           [-1, 64, 12, 12]               0
         MaxPool2d-4             [-1, 64, 6, 6]               0
            Conv2d-5             [-1, 64, 6, 6]           4,096
       BatchNorm2d-6             [-1, 64, 6, 6]             128
              ReLU-7             [-1, 64, 6, 6]               0
            Conv2d-8             [-1, 64, 6, 6]          36,864
       BatchNorm2d-9             [-1, 64, 6, 6]             128
             ReLU-10             [-1, 64, 6, 6]               0
           Conv2d-11            [-1, 256, 6, 6]          16,384
      BatchNorm2d-12            [-1, 256, 6, 6]             512
           Conv2d-13            [-1, 256, 6, 6]          16,384
      BatchNorm2d-14            [-1, 25

Hyper-Parameters
-

In [7]:
# Updated Hyper-parameters
in_channels =3
learning_rate = 3e-4 #karpathy's constant
num_epochs = 3
batch_size =16

Load the Data
-

Input Size, Conv Base Output Calculation,   Final Feature Map (Channels * H * W)
1. 32,       |      32/32=1,             |           512×1×1=512
2. 75,       |    75/32=2.34→2,          |           512×2×2=2048

In [8]:
trans_train = transforms.Compose([transforms.Resize(150), 
                                  transforms.RandomRotation(30),
                                  transforms.RandomResizedCrop(32),
                                  transforms.RandomHorizontalFlip(),
                                  transforms. ToTensor()])

data_dir = "C:/Users/dbda.STUDENTSDC/Desktop/Practical Machine Learning/SheepFaceImages"
full_dataset = datasets.ImageFolder(data_dir, transform=trans_train)
dataset_size =len(full_dataset)
train_size = int(0.8*dataset_size)
test_size = dataset_size - train_size
train_dataset, test_dataset = random_split(full_dataset,[train_size,test_size])
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle= True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle =False)
type(train_dataset)

torch.utils.data.dataset.Subset

Loss and optimizer
-

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Train Network
--

In [10]:
for epoch in range(num_epochs):
    loop = tqdm(train_loader, leave=True)
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device)
        targets = targets.to(device) # Keep as integers for CrossEntropy

        # forward
        scores = model(data)
        loss = criterion(scores, targets) # No one_hot needed here!
        
        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_description(f"Epoch: [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

Epoch: [3/3]: 100%|██████████| 84/84 [00:42<00:00,  1.97it/s, loss=0.949]


##### Check accuracy on training & test to see how good our model

In [11]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)
            # predictions = (scores[:,1]>0.5).long()
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)
        
    model.train()
    return float(num_correct) / num_samples


In [12]:
# Check accuracy on both sets
train_acc = check_accuracy(train_loader, model)
test_acc = check_accuracy(test_loader, model)

print(f"Accuracy on training set: {train_acc*100:.2f}%")
print(f"Accuracy on test set: {test_acc*100:.2f}%")

Accuracy on training set: 48.66%
Accuracy on test set: 46.73%


Resnet18

Weights = DEFAULT
1. Accuracy on training set: 54.61%
2. Accuracy on test set: 53.87%

Weights = IMAGENET1K_V1
1. Accuracy on training set: 62.65%
2. Accuracy on test set: 57.14%

Resnet50 & Weights = IMAGENET1K_V1
1. Accuracy on training set: 48.66%
2. Accuracy on test set: 46.73%